In [1]:
import sys
sys.path.append("../src")

In [2]:
from Fasttext.Connect_Database import connect
from preprocessing import clean,stopword
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split
import csv

In [3]:
app = connect()
df = pd.DataFrame(app.load_data_VnExpress())

In [4]:
df[['topic', 'subtopic', 'content']].head(10)

,topic,subtopic,content
0,Thời sự,Chính trị,"Chiều 19/8, Tổng Bí thư Tô Lâm làm việc với Ba..."
1,Thời sự,Chính trị,"Tổng Bí thư nhấn mạnh việc chuẩn bị, tổ chức c..."
2,Thời sự,Chính trị,Người đứng đầu Đảng đề nghị các cơ quan triển ...
3,Thời sự,Chính trị,"Đồng thời, Tổng Bí thư lưu ý công tác đón tiếp..."
4,Thời sự,Chính trị,"Tổng Bí thư lưu ý trong dịp này, đặc biệt là v..."
5,Thời sự,Chính trị,Theo báo cáo của Ban Tuyên giáo và Dân vận Tru...
6,Thời sự,Chính trị,Được xây dựng ở giai đoạn ác liệt nhất cuộc ch...
7,Thời sự,Chính trị,"Trung tá Nguyễn Hồng Giang, Phân đội trưởng Ch..."
8,Thời sự,Chính trị,"Để đối phó thời tiết mưa nhiều, Đội tranh thủ ..."
9,Thời sự,Chính trị,Ba sĩ quan thuộc Cục Gìn giữ hòa bình Việt Nam...


In [5]:
def process_texts(texts):
    cleaned = [clean(str(t)) for t in texts]
    processed = stopword(cleaned)
    return processed

In [6]:
df['clean_content'] = process_texts(df['content'])

In [7]:
df['label'] = '__label__' + df['topic'].str.replace(' ', '_') + '__' + df['subtopic'].str.replace(' ', '_')

In [8]:
df.head(10)

,content,subtopic,title,topic,clean_content,label
0,"Chiều 19/8, Tổng Bí thư Tô Lâm làm việc với Ba...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,chiều 19 8 tổng bí thư tô lâm ban đạo trung ươ...,__label__Thời_sự__Chính_trị
1,"Tổng Bí thư nhấn mạnh việc chuẩn bị, tổ chức c...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,tổng bí thư nhấn chuẩn tổ chức hoạt động kỷ ni...,__label__Thời_sự__Chính_trị
2,Người đứng đầu Đảng đề nghị các cơ quan triển ...,Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,đứng đầu đảng đề nghị quan triển khai hiệu kế ...,__label__Thời_sự__Chính_trị
3,"Đồng thời, Tổng Bí thư lưu ý công tác đón tiếp...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,đồng thời tổng bí thư lưu công tác đón tiếp đạ...,__label__Thời_sự__Chính_trị
4,"Tổng Bí thư lưu ý trong dịp này, đặc biệt là v...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,tổng bí thư lưu dịp đặc biệt du đổ thủ đô hà n...,__label__Thời_sự__Chính_trị
5,Theo báo cáo của Ban Tuyên giáo và Dân vận Tru...,Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,báo cáo ban tuyên giáo dân vận trung ương công...,__label__Thời_sự__Chính_trị
6,Được xây dựng ở giai đoạn ác liệt nhất cuộc ch...,Chính trị,Tặng kỷ vật đường ống xăng dầu Trường Sơn cho ...,Thời sự,xây dựng giai đoạn ác liệt chiến chống mỹ hứng...,__label__Thời_sự__Chính_trị
7,"Trung tá Nguyễn Hồng Giang, Phân đội trưởng Ch...",Chính trị,Bộ đội Việt Nam xây kho hậu cần chiến lược cho...,Thời sự,trung tá nguyễn hồng giang phân đội trưởng huy...,__label__Thời_sự__Chính_trị
8,"Để đối phó thời tiết mưa nhiều, Đội tranh thủ ...",Chính trị,Bộ đội Việt Nam xây kho hậu cần chiến lược cho...,Thời sự,đối phó thời tiết mưa đội tranh thủ nắng trưa ...,__label__Thời_sự__Chính_trị
9,Ba sĩ quan thuộc Cục Gìn giữ hòa bình Việt Nam...,Chính trị,Bộ đội Việt Nam xây kho hậu cần chiến lược cho...,Thời sự,sĩ quan cục gìn hòa bình việt nam tổng cục hậu...,__label__Thời_sự__Chính_trị


In [9]:
df['format'] = df['label'] + ' ' + df['clean_content']
max_count= 2000
balanced_data = (df.groupby('label', group_keys=False, as_index=False).apply(lambda x: x.sample(max_count, replace=True)).reset_index(drop=True))

train, test = train_test_split(balanced_data, test_size=0.2, random_state=42)
train['format'].to_csv("../Data/processed/dataset_train.txt",index=False, header=False,sep='\n', quoting=csv.QUOTE_NONE, escapechar='\\')
test['format'].to_csv("../Data/processed/dataset_test.txt",index=False, header=False,sep='\n', quoting=csv.QUOTE_NONE, escapechar='\\')
df['format'].to_csv('../Data/processed/train.txt', index=False, header=False, sep='\n',quoting=csv.QUOTE_NONE,escapechar='\\')

C:\Users\admin\AppData\Local\Temp\ipykernel_8836\1075786500.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_data = (df.groupby('label', group_keys=False, as_index=False).apply(lambda x: x.sample(max_count, replace=True)).reset_index(drop=True))


In [10]:
pd.set_option('display.max_rows', None)
print(balanced_data['label'].value_counts())

label
__label__Bất_động_sản__Chính_sách                  2000
__label__Thế_giới__Người_Việt_5_châu               2000
__label__Thể_thao__Europa_League                   2000
__label__Thể_thao__Các_môn_khác                    2000
__label__Thể_thao__Các_giải_khác                   2000
__label__Thể_thao__Champions_League                2000
__label__Thể_thao__Bóng_đá                         2000
__label__Thể_thao__Bundesliga                      2000
__label__Thế_giới__Tư_liệu                         2000
__label__Thế_giới__Quân_sự                         2000
__label__Thế_giới__Phân_tích                       2000
__label__Thế_giới__Cuộc_sống_đó_đây                2000
__label__Thể_thao__La_Liga                         2000
__label__Thế_giới__Bắc_Mỹ                          2000
__label__Thư_giãn__Đố_vui                          2000
__label__Thư_giãn__Trò_chơi                        2000
__label__Thư_giãn__Thú_cưng                        2000
__label__Thư_giãn__Cười                   

In [11]:
model = fasttext.train_supervised(
    input="../Data/processed/dataset_train.txt",
    lr=1.0,
    epoch=100,
    wordNgrams=3,
    dim=100,
    loss='softmax'
)

In [12]:
result = model.test("../Data/processed/dataset_test.txt")
print(f"Số mẫu test: {result[0]}")
print(f"Độ chính xác (precision): {result[1]:.4f}")
print(f"Độ bao phủ (recall): {result[2]:.4f}")
print(f"F1-score: {2 * result[1] * result[2] / (result[1] + result[2]):.4f}")

Số mẫu test: 38000
Độ chính xác (precision): 0.9156
Độ bao phủ (recall): 0.9156
F1-score: 0.9156


In [14]:
model.save_model("../models/Fasttext/model.bin")

In [15]:
df_Reddit = pd.DataFrame(app.load_data())
df_Reddit['clean_content'] = process_texts(df_Reddit['content'])

In [16]:
df_Reddit.head(5)

,link,author,comments,content,date,image_link,subreddit,title,upvotes,video_link,clean_content
0,https://www.reddit.com/r/TroChuyenLinhTinh/com...,Friendly-Lie5849,37,Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoa...,2025-08-22 07:25:50,None,TroChuyenLinhTinh,Những gì đang diễn ra ở Trung Quốc đang dần lặ...,157,None,phim tuyên truyền chủ nghĩa dân tộc cực đoan m...
1,https://www.reddit.com/r/vozforums/comments/1m...,TinNT0409,1,"Chào mọi người, đợt này mình có dùng Mathpix đ...",2025-08-22 07:23:33,https://external-preview.redd.it/kqhxR96oyl5gx...,vozforums,Phần mềm quét công thức sang Word miễn phí tha...,2,None,chào đợt mathpix quét công thức đc 10 lượt viế...
2,https://www.reddit.com/r/TroChuyenLinhTinh/com...,Weekly_Top7078,3,Trước kì kinh bao nhiêu ngày thì an toàn?,2025-08-22 07:07:46,None,TroChuyenLinhTinh,Ngày an toàn,0,None,kì kinh bao nhiêu an toàn
3,https://www.reddit.com/r/VietNamNation/comment...,Ambitious-Fan-9831,18,ờm bỏ qua tỉ lệ gái đẹp ra thì tao thấy bề nổi...,2025-08-22 07:04:09,None,VietNamNation,Thái Lan có phải là 1 hình mẫu nước đáng sống ...,16,None,ờm tỉ lệ gái đẹp tao bề nổi thái lọ cánh tả số...
4,https://www.reddit.com/r/vozforums/comments/1m...,Nicklas0704,3,"Hi các bác, e làm mmo có dư 1 ít, cụ thể là kh...",2025-08-22 14:03:21,None,vozforums,Hiện tại nên bỏ tiền vào đâu ?,0,None,hi e mmo dư 1 cụ thể 10 tỉ tham gia đầu hiện e...


In [17]:
df_Reddit = df_Reddit[
    df_Reddit['clean_content'].notna() &                   
    (df_Reddit['clean_content'].str.strip().str.lower() != 'none') &
    (df_Reddit['clean_content'].str.strip() != '')
]

In [18]:
df_Reddit = df_Reddit[['author', 'title','clean_content', 'content']]
df_Reddit.head(2)

,author,title,clean_content,content
0,Friendly-Lie5849,Những gì đang diễn ra ở Trung Quốc đang dần lặ...,phim tuyên truyền chủ nghĩa dân tộc cực đoan m...,Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoa...
1,TinNT0409,Phần mềm quét công thức sang Word miễn phí tha...,chào đợt mathpix quét công thức đc 10 lượt viế...,"Chào mọi người, đợt này mình có dùng Mathpix đ..."


In [19]:
labels = []
scores = []
for text in df_Reddit['clean_content']:
    if not isinstance(text, str):
        text = ""
    text_clean = text.replace("\n", " ").strip()
    prediction = model.predict(text_clean)
    labell = prediction[0][0].replace("__label__", "")
    score = prediction[1][0]
    labels.append(labell)
    scores.append(score)
df_Reddit['label'] = labels
df_Reddit['score'] = scores


In [20]:
df_Reddit.head(10)

,author,title,clean_content,content,label,score
0,Friendly-Lie5849,Những gì đang diễn ra ở Trung Quốc đang dần lặ...,phim tuyên truyền chủ nghĩa dân tộc cực đoan m...,Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoa...,Giải_trí__Phim,0.235065
1,TinNT0409,Phần mềm quét công thức sang Word miễn phí tha...,chào đợt mathpix quét công thức đc 10 lượt viế...,"Chào mọi người, đợt này mình có dùng Mathpix đ...",Đời_sống__Bài_học_sống,0.941172
2,Weekly_Top7078,Ngày an toàn,kì kinh bao nhiêu an toàn,Trước kì kinh bao nhiêu ngày thì an toàn?,Thư_giãn__Đố_vui,0.609811
3,Ambitious-Fan-9831,Thái Lan có phải là 1 hình mẫu nước đáng sống ...,ờm tỉ lệ gái đẹp tao bề nổi thái lọ cánh tả số...,ờm bỏ qua tỉ lệ gái đẹp ra thì tao thấy bề nổi...,Thế_giới__Tư_liệu,0.306790
4,Nicklas0704,Hiện tại nên bỏ tiền vào đâu ?,hi e mmo dư 1 cụ thể 10 tỉ tham gia đầu hiện e...,"Hi các bác, e làm mmo có dư 1 ít, cụ thể là kh...",Kinh_doanh__Tiền_của_tôi,0.624146
5,Chunghiacanhanvidai,Đừng Đánh Giá Tự Do Xã Hội Bằng Giáo Điều Phươ...,đừng đánh giá xã hội giáo phương tây trần trụi...,Đừng Đánh Giá Tự Do Xã Hội Bằng Giáo Điều Phươ...,Giải_trí__Sách,0.575920
7,117431853211,Xin mọi người cho em lời khuyên,17 sống bình chí thể coi êm đềm bất kỳ biến cố...,"Năm nay em 17 tuổi, cuộc sống rất bình thường,...",Đời_sống__Bài_học_sống,0.176320
8,LeeTuneVane,Con BHP rồi mẹ ạ .Nếu may mắn thì gia đình mìn...,bhp mẹ may mắn gia đình kinh tế mẹ bát cơm can...,Con BHP rồi mẹ ạ .Nếu may mắn thì gia đình mìn...,Đời_sống__Bài_học_sống,0.488035
10,dahoodcashseller,Sỹ con chạy xe quá lẹ bắt chước cha,welp ae kết tụi chạy 70km h ko đội nón tông xo...,Welp chắc ae cũng biết cái kết rồi tụi này chạ...,Thể_thao__Các_giải_khác,0.198375
11,fishmeaterm,Kêu gọi toàn dân VietNamNation vào trị bọn Pod...,kêu gọi toàn dân đảo vietnamnation trị bọn pod...,Kêu gọi toàn dân đảo VietNamNation vào trị bọn...,Kinh_doanh__NetZero,0.740406


In [23]:
for idx, row in df_Reddit.head(50).iterrows():
    print(f"Author: {row['author']}")
    print(f"Title: {row['title']}")
    print(f"Content: {row['content'][:200]}...")
    print(f"Label: {row['label']}")
    print(f"Score: {row['score']:.3f}")
    print("-"*60)

Author: Friendly-Lie5849
Title: Những gì đang diễn ra ở Trung Quốc đang dần lặp lại ở Việt Nam
Content: Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoan Mưa Đỏ aka Máu Kinh đang được truyền thông, dư luận viên, bò đỏ, Sô-Vanh con tung hê hết lời. Phim hiện đã thu về 28 tỷ đồng vào ngày đầu công chiếu là ...
Label: Giải_trí__Phim
Score: 0.235
------------------------------------------------------------
Author: TinNT0409
Title: Phần mềm quét công thức sang Word miễn phí thay thế Mathpix, Mathtype, Equation mặc định trong Word
Content: Chào mọi người, đợt này mình có dùng Mathpix để quét công thức nhưng đc có 10 lượt nên Viết luôn app thay thế miễn phí [AuraLateX](https://auravsoftware.com/chuyen-cong-thuc-toan-sang-word/)...
Label: Đời_sống__Bài_học_sống
Score: 0.941
------------------------------------------------------------
Author: Weekly_Top7078
Title: Ngày an toàn
Content: Trước kì kinh bao nhiêu ngày thì an toàn?...
Label: Thư_giãn__Đố_vui
Score: 0.610
------------------------------